In [ ]:
import marimo as mo

# MLモデルのデプロイ ハンズオン

このノートブックでは、06で作った **SAM (Segment Anything Model)** を
**FastAPI + カスタムコンテナ** で Vertex AI にデプロイする流れを体験します。

## 全体の流れ

```
Step 0: モデルをGCSにアップロード（06の成果物）
  ↓
Step 1: FastAPIサーバのコード確認
  ↓
Step 2: ローカルでDockerテスト
  ↓
Step 3: Artifact Registryにコンテナをpush
  ↓
Step 4: Vertex AIにデプロイ（gcloud CLI）
  ↓
Step 5: エンドポイントに推論リクエスト（Python SDK）
  ↓
[拡張] Step 6: A/Bテスト（トラフィック分割）
[拡張] Step 7: Gradioデモアプリ
```

## 今日のキーコンセプト：モデルとコンテナを分離する

| ❌ アンチパターン | ✅ 今回学ぶパターン |
|---|---|
| モデルをDockerイメージに焼き込む | モデルをGCSに保管 |
| モデル更新 → コンテナリビルド必要 | モデル更新だけで済む |
| コンテナサイズが肥大化 | コンテナは軽量 |

## セットアップ

In [ ]:
# --- TODO: 自分の名前（英字小文字）を入れてください ---
USER = "___"

if USER == "your_name":
    raise ValueError("USER を自分の名前（英字小文字）に変更してください！")

PROJECT_ID = "hr-mixi"
REGION = "asia-northeast1"
GCS_BUCKET = "hr-mixi-ml-hands-on"

MODEL_GCS_URI = f"gs://{GCS_BUCKET}/2026/models/{USER}/sam-model/"
IMAGE_URI = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/ml-hands-on/sam-server:{USER}"

print(f"USER          : {USER}")
print(f"MODEL_GCS_URI : {MODEL_GCS_URI}")
print(f"IMAGE_URI     : {IMAGE_URI}")

---

## Step 0: モデルを GCS にアップロード

### なぜ GCS にモデルを置くのか？

MLOps の定石として、**モデルとサービングコードを分離**します。

- モデルファイル（数百MB）をコンテナに含めると、更新のたびに `docker build → push` が必要
- GCS に置くことで、モデルの更新はファイルのアップロードだけで完結
- コンテナはサービングロジックのみ管理 → 責務の明確化

> Vertex AI も公式にこのパターンを推奨しています。モデル登録時に `artifact_uri` を指定すると、
> Vertex AI がモデルを管理バケットにコピーし、コンテナに `AIP_STORAGE_URI` 環境変数（`gs://` URI）を自動設定します。
> コンテナのデフォルト SA にはこの URI への読み取り権限が自動付与されるため、権限設定が不要です。

### SAM モデルの保存

まず、06 で使った SAM モデルを `save_pretrained()` で保存します。
これにより `sam-vit-base/` ディレクトリに `config.json`, `model.safetensors` 等が保存されます。

In [ ]:
from transformers import SamModel, SamProcessor

model = SamModel.from_pretrained("facebook/sam-vit-base")
processor = SamProcessor.from_pretrained("facebook/sam-vit-base")

model.save_pretrained("sam-vit-base")
processor.save_pretrained("sam-vit-base")

In [ ]:
mo.md(f"""
保存したモデルディレクトリを GCS にアップロードしてください：

```bash
gcloud storage cp -r sam-vit-base/* {MODEL_GCS_URI}
```

アップロードできたか確認：

```bash
gcloud storage ls {MODEL_GCS_URI}
```
""")

---

## Step 1: FastAPI サーバのコード確認

### コードの確認

- src/predictor.py
- src/app.py

### 設計のポイント

コンテナ起動時（`lifespan`）に GCS からモデルをダウンロードします。
モデルはコンテナに含まれていません。

```python
@asynccontextmanager
async def lifespan(app: FastAPI):
    if AIP_STORAGE_URI:
        # Vertex AI が管理バケットにコピーした URI を使用
        gcs_uri = AIP_STORAGE_URI
    elif MODEL_GCS_URI:
        # ローカル Docker テスト用フォールバック
        gcs_uri = MODEL_GCS_URI
    download_model(gcs_uri, LOCAL_MODEL_DIR)
    app.state.predictor = SAMPredictor(LOCAL_MODEL_DIR)
    yield
```

- Vertex AI 上では `AIP_STORAGE_URI` が自動設定される（`artifact_uri` 経由）
- ローカルテスト時は `MODEL_GCS_URI` 環境変数で GCS URI を直接渡す
- `/health` エンドポイント → Vertex AI のヘルスチェックに必要
- `/predict` エンドポイント → 画像 + ポイント座標を受け取り、セグメンテーションマスクをJSONで返す

### API の入出力

**リクエスト:**
```json
{
  "instances": [
    {
      "image": "<base64 画像>",
      "input_points": [[x, y]],
      "input_labels": [1]
    }
  ]
}
```

**レスポンス:**
```json
{
  "predictions": [
    {
      "mask_b64": "<base64 PNG マスク画像>",
      "iou_score": 0.95
    }
  ]
}
```

---

## Step 2: ローカルで Docker テスト

### Dockerfile の確認

`Dockerfile` を確認して、モデルはコンテナに含まれていないことを確認してください。

In [ ]:
mo.md(f"""
### Docker ビルドと起動

以下のコマンドをターミナルで実行してください：

```bash
# 1. イメージをビルド
sudo docker build -t sam-server --platform linux/amd64 .
```

#### Mac（ローカル）の場合

`~/.config/gcloud` の認証情報をマウントして使います。

```bash
sudo docker run --platform linux/amd64 -p 8081:8080 \\
    -e MODEL_GCS_URI="{MODEL_GCS_URI}" \\
    -e GOOGLE_CLOUD_PROJECT="{PROJECT_ID}" \\
    -v ~/.config/gcloud:/root/.config/gcloud:ro \\
    sam-server
```

#### Vertex AI Workbench の場合

GCE メタデータサーバー経由で認証するため、`--network host` を指定します。
`~/.config/gcloud` のマウントは不要です。

```bash
sudo docker run --network host \\
    -e MODEL_GCS_URI="{MODEL_GCS_URI}" \\
    -e GOOGLE_CLOUD_PROJECT="{PROJECT_ID}" \\
    sam-server \\
    uv run uvicorn src.app:app --host 0.0.0.0 --port 8081
```

> `--network host` ではコンテナがホストのネットワークを直接共有するため、`-p` によるポートマッピングは無効です。
> Workbench では 8080 が JupyterLab 等に使われているため、CMD を上書きしてポートを 8081 に変更しています。

---

別のターミナルで動作確認：

```bash
# ヘルスチェック
curl http://localhost:8081/health
# 期待する結果: {{"status": "ok"}}

# 推論テスト（base64 JSON形式、-d @- でstdinから渡す）
IMAGE_B64=$(base64 -w0 ./images/sample.jpeg)
echo '{{"instances": [{{"image": "'"$IMAGE_B64"'", "input_points": [[1300, 400]], "input_labels": [1]}}]}}' | \\
    curl -X POST http://localhost:8081/predict \\
        -H "Content-Type: application/json" \\
        -d @-
```
""")

### ローカルテストの確認

以下を実行して、正常に応答が返ってくることを確認してください：

- `/health` → `{"status": "ok"}` が返る
- `/predict` → `{"predictions": [{"mask_b64": "...", "iou_score": ...}]}` 形式の JSON が返る

問題なければ次のステップへ進みましょう。

### ローカルテストの可視化

curl だと `mask_b64` が長い文字列で結果がわかりません。
以下のセルを実行すると、ローカルサーバに Python でリクエストを送り、
マスクを元画像にオーバーレイして表示します。

> **注意**: Docker コンテナが `localhost:8081` で起動中であることを確認してください。

In [ ]:
import base64 as _b64
import io as _io
import pathlib as _pathlib

import numpy as _np
import requests as _requests
from PIL import Image as _Image

# サンプル画像のパス
_sample_path = _pathlib.Path("images/sample.jpeg")

# 画像を読み込み base64 エンコード
_img = _Image.open(_sample_path).convert("RGB")
_buf = _io.BytesIO()
_img.save(_buf, format="JPEG")
_image_b64 = _b64.b64encode(_buf.getvalue()).decode("utf-8")

# sample.jpeg (2118x718) の中央の人物（ピンクの服）の胴体付近を指定
# 画像中心だと人物の隙間に当たりセグメントされないため、固定座標を使用
_cx, _cy = 1300, 400

# ローカルサーバにリクエスト送信
_resp = _requests.post(
    "http://localhost:8081/predict",
    json={
        "instances": [
            {
                "image": _image_b64,
                "input_points": [[_cx, _cy]],
                "input_labels": [1],
            }
        ]
    },
    timeout=60,
)
_resp.raise_for_status()
_pred = _resp.json()["predictions"][0]

# マスクをデコードしてオーバーレイ
_mask_bytes = _b64.b64decode(_pred["mask_b64"])
_mask_image = _Image.open(_io.BytesIO(_mask_bytes))
_mask_array = _np.array(_mask_image)
_img_array = _np.array(_img)
_overlay = _img_array.copy()
_mask_pixels = int(_np.sum(_mask_array > 0))
_total_pixels = _mask_array.shape[0] * _mask_array.shape[1]
_overlay[_mask_array > 0] = (
    _overlay[_mask_array > 0] * 0.5 + _np.array([30, 144, 255]) * 0.5
).astype(_np.uint8)
_result_image = _Image.fromarray(_overlay)

mo.vstack(
    [
        mo.md(
            f"""
### ローカルテスト推論結果

- ポイント座標: ({_cx}, {_cy})
- マスク信頼度スコア: **{_pred["iou_score"]:.3f}**（SAM が自己評価したマスク品質。Ground Truth との IoU ではない）
- マスク領域: {_mask_pixels:,} / {_total_pixels:,} ピクセル ({_mask_pixels / _total_pixels * 100:.1f}%)
"""
        ),
        mo.hstack(
            [
                mo.vstack(
                    [
                        mo.md("**元画像 + マスクオーバーレイ**"),
                        mo.image(_result_image),
                    ]
                ),
                mo.vstack(
                    [
                        mo.md("**マスク画像（白=セグメント領域）**"),
                        mo.image(_mask_image),
                    ]
                ),
            ],
            justify="start",
        ),
    ]
)

---

## Step 3: Artifact Registry にコンテナを push

ビルドしたイメージを Google Cloud の **Artifact Registry** に push します。
Vertex AI はここからコンテナイメージを取得してデプロイします。

In [ ]:
mo.md(f"""
```bash
# Docker の認証設定（Artifact Registry に push するために必要）
gcloud auth configure-docker asia-northeast1-docker.pkg.dev

# タグをつける
sudo docker tag sam-server {IMAGE_URI}

# push
sudo docker push {IMAGE_URI}
```

push が完了したら確認：

```bash
gcloud artifacts docker images list \\
    asia-northeast1-docker.pkg.dev/hr-mixi/ml-hands-on \\
    --filter="package=sam-server"
```
""")

---

## Step 4: Vertex AI にデプロイ

### 4-1. モデルをモデルレジストリに登録

コンテナイメージと `artifact-uri`（モデルの GCS パス）を組み合わせて登録します。
`artifact-uri` を指定すると、Vertex AI がモデルを管理バケットにコピーし、
コンテナに `AIP_STORAGE_URI` 環境変数を自動設定します。

In [ ]:
mo.md(f"""
```bash
gcloud ai models upload \\
    --region={REGION} \\
    --display-name=sam-server-{USER} \\
    --artifact-uri={MODEL_GCS_URI} \\
    --container-image-uri={IMAGE_URI} \\
    --container-health-route=/health \\
    --container-predict-route=/predict \\
    --container-ports=8080
```

コマンド実行後、出力された `MODEL_ID` をメモしておいてください。

もし `MODEL_ID` を忘れた場合は、以下のコマンドで確認できます：

```bash
gcloud ai models list \\
    --region={REGION} \\
    --filter="displayName=sam-server-{USER}"
```
""")

### 4-2. エンドポイントの作成

In [ ]:
mo.md(f"""
```bash
gcloud ai endpoints create \\
    --region={REGION} \\
    --display-name=sam-endpoint-{USER}
```

コマンド実行後、出力された `ENDPOINT_ID` をメモしておいてください。

もし `ENDPOINT_ID` を忘れた場合は、以下のコマンドで確認できます：

```bash
gcloud ai endpoints list \\
    --region={REGION} \\
    --filter="displayName=sam-endpoint-{USER}"
```
""")

### 4-3. エンドポイントにモデルをデプロイ

In [ ]:
# --- TODO: gcloud コマンドの出力から ID を入力してください ---
ENDPOINT_ID = "___"  # TODO: gcloud ai endpoints create の出力から
MODEL_ID = "___"  # TODO: gcloud ai models upload の出力から

In [ ]:
mo.md(f"""
```bash
gcloud ai endpoints deploy-model {ENDPOINT_ID} \\
    --region={REGION} \\
    --model={MODEL_ID} \\
    --display-name=sam-server-{USER} \\
    --machine-type=n1-standard-2
```

> デプロイには **5〜10分** かかります。
> `gcloud ai endpoints describe {ENDPOINT_ID} --region={REGION}` でステータスを確認できます。
""")

---

## Step 5: エンドポイントへの推論テスト

Vertex AI Python SDK を使って、デプロイしたエンドポイントに推論リクエストを送ります。
画像とポイント座標を送ると、セグメンテーションマスクが返ってきます。

In [ ]:
import base64

from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)

if ENDPOINT_ID == "___":
    raise RuntimeError("TODO: 上のセルで ENDPOINT_ID を設定してください。")

mo.md(f"Vertex AI SDK 初期化完了（project={PROJECT_ID}, region={REGION}）")

In [ ]:
import io
import pathlib

import numpy as np
from PIL import Image

sample_image_path = pathlib.Path("images/sample.jpeg")

if ENDPOINT_ID == "___":
    raise RuntimeError("ENDPOINT_ID を設定してから実行してください。")

endpoint = aiplatform.Endpoint(ENDPOINT_ID)

with open(sample_image_path, "rb") as img_f:
    image_b64 = base64.b64encode(img_f.read()).decode("utf-8")

cx, cy = 1300, 400

response = endpoint.predict(
    instances=[
        {
            "image": image_b64,
            "input_points": [[cx, cy]],
            "input_labels": [1],
        }
    ]
)

# マスク画像をデコードして表示
pred = response.predictions[0]
mask_bytes = base64.b64decode(pred["mask_b64"])
mask_image = Image.open(io.BytesIO(mask_bytes))
mask_array = np.array(mask_image)

# 元画像にマスクをオーバーレイ
img_array = np.array(Image.open(sample_image_path).convert("RGB"))
overlay = img_array.copy()
overlay[mask_array > 0] = (
    overlay[mask_array > 0] * 0.5 + np.array([30, 144, 255]) * 0.5
).astype(np.uint8)
result_image = Image.fromarray(overlay)

mo.vstack(
    [
        mo.md(
            f"""
### 推論結果

- ポイント座標: ({cx}, {cy})
- マスク信頼度スコア: **{pred["iou_score"]:.3f}**
"""
        ),
        mo.image(result_image),
    ]
)

---

## Step 6: A/B テスト（トラフィック分割）

### なぜ A/B テストが必要か

- オフライン指標が良くても、**本番データでの挙動は別**
- 新モデルをいきなり100%に切り替えるのはリスクが高い
- 少量のトラフィックで本番環境の動作を確認してから段階的に移行する

### 段階的なロールアウト

```
新モデル20% ─→ 新モデル50% ─→ 新モデル100%
           測定             測定
```

各段階で「エラー率」「レイテンシ」「セグメンテーション品質」を確認します。

### Vertex AI でのトラフィック分割

Vertex AI のエンドポイントは**複数のモデルを同時にデプロイ**できます。

In [ ]:
mo.md(f"""
```bash
# v2 モデルを GCS にアップロード（再学習モデルや別バリアント）
gcloud storage cp -r sam-vit-base-v2/* gs://{GCS_BUCKET}/2026/models/{USER}/sam-model-v2/

# v2 をモデルレジストリに登録
gcloud ai models upload \\
    --region={REGION} \\
    --display-name=sam-server-v2-{USER} \\
    --artifact-uri=gs://{GCS_BUCKET}/2026/models/{USER}/sam-model-v2/ \\
    --container-image-uri={IMAGE_URI} \\
    --container-health-route=/health \\
    --container-predict-route=/predict \\
    --container-ports=8080

MODEL_V2_ID=___  # TODO: 上記コマンドの出力から

# v1 を 80%、v2 を 20% にトラフィック分割
gcloud ai endpoints deploy-model {ENDPOINT_ID} \\
    --region={REGION} \\
    --model=$MODEL_V2_ID \\
    --display-name=sam-model-v2 \\
    --traffic-split=0=80,$MODEL_V2_DEPLOYMENT_ID=20 \\
    --machine-type=n1-standard-2
```

同じエンドポイントに送るだけで、自動的に v1/v2 に振り分けられます。
""")

---

## Step 7（拡張）: Gradio デモアプリ

Vertex AI エンドポイントに接続した **Gradio** デモを作ります。
画像をアップロードしてクリックすると、その箇所のセグメンテーション結果を表示します。

In [ ]:
mo.md(f"""
ターミナルで以下のコマンドを実行して Gradio デモを起動します：

```bash
uv run python scripts/gradio_demo.py \\
    --project {PROJECT_ID} \\
    --region {REGION} \\
    --endpoint {ENDPOINT_ID} \\
    --share
```

ブラウザで表示される URL にアクセスし、画像をアップロードしてクリックするとセグメンテーション結果が表示されます。
""")

---

## まとめ

今日体験したこと：

| ステップ | 学んだこと |
|---|---|
| Step 0 | GCSにモデルを置く → MLOpsのバージョン管理 |
| Step 1 | FastAPI + lifespan でGCSからモデルをDL |
| Step 2 | Dockerカスタムコンテナ（モデルを含まない設計） |
| Step 3 | Artifact Registry へのコンテナ管理 |
| Step 4 | Vertex AI へのデプロイ（モデル登録 → エンドポイント） |
| Step 5 | Python SDK での推論リクエスト |
| Step 6 | トラフィック分割でリスクを抑えたモデル更新 |

### 重要な設計原則

> **「モデルとコンテナを分離する」**
>
> - コンテナ：サービングロジックのみ（再利用性が高い）
> - モデル：GCSで管理（更新が独立している）
> - `artifact_uri` + `AIP_STORAGE_URI`：Vertex AI 推奨のモデル受け渡しパターン